# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [52]:
import os
import re
import time
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

for path in ["sql", "reports", "screenshots"]:
    Path(path).mkdir(exist_ok=True)

DB_HOST = os.getenv("QBC12_DB_HOST", "SERVERIP")
DB_PORT = os.getenv("QBC12_DB_PORT", "32112")
DB_NAME = os.getenv("QBC12_DB_NAME", "qbc12_airbnb")
DB_USER = os.getenv("QBC12_DB_USER", "") or input("DB user: ").strip()
DB_PASSWORD = os.getenv("QBC12_DB_PASSWORD", "") or input("DB password: ").strip()
STUDENT_ID = os.getenv("QBC12_STUDENT_ID", "") or DB_USER.replace("student_", "")

safe_student = re.sub(r"[^a-zA-Z0-9_]", "_", STUDENT_ID.lower())
STUDENT_SCHEMA = f"student_{safe_student}"

url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text("SELECT version()")).scalar()

print("Schema:", STUDENT_SCHEMA)
print("Postgres:", version[:100])

Schema: student_2109
Postgres: PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19


## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [53]:
columns_sql = '''
select table_schema, table_name, column_name, data_type
from information_schema.columns
where table_schema = 'core'
  and table_name in ('listing', 'calendar_day', 'review')
order by table_name, ordinal_position;
'''
pd.read_sql(columns_sql, engine)

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [54]:
row_count_sql = '''
select 'core.listing' as table_name, count(*) as rows from core.listing
union all select 'core.calendar_day', count(*) from core.calendar_day
union all select 'core.review', count(*) from core.review;
'''
pd.read_sql(row_count_sql, engine)

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [73]:
STUDENT_SCHEMA = "student_ayda_hafezian"

In [74]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

def schema_exists(schema_name: str) -> bool:
    with engine.connect() as conn:
        return conn.execute(
            text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.schemata
                    WHERE schema_name = :schema_name
                )
            """),
            {"schema_name": schema_name},
        ).scalar()

assert schema_exists(STUDENT_SCHEMA), f"Schema {STUDENT_SCHEMA} does not exist"
print("Schema ready:", STUDENT_SCHEMA)

Schema ready: student_ayda_hafezian


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [75]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = """
SELECT
    c.listing_id,
    AVG(c.price) FILTER (WHERE c.price IS NOT NULL) AS avg_calendar_price_30,
    AVG(CASE WHEN c.available THEN 1.0 ELSE 0.0 END) AS availability_30_rate
FROM core.calendar_day c
WHERE c.date >= CURRENT_DATE
  AND c.date < CURRENT_DATE + INTERVAL '30 days'
GROUP BY c.listing_id
"""
print(calendar_30_sql)


SELECT
    c.listing_id,
    AVG(c.price) FILTER (WHERE c.price IS NOT NULL) AS avg_calendar_price_30,
    AVG(CASE WHEN c.available THEN 1.0 ELSE 0.0 END) AS availability_30_rate
FROM core.calendar_day c
WHERE c.date >= CURRENT_DATE
  AND c.date < CURRENT_DATE + INTERVAL '30 days'
GROUP BY c.listing_id



In [76]:
pd.read_sql(f"""
SELECT *
FROM ({calendar_30_sql}) t
LIMIT 10
""", engine)

,listing_id,avg_calendar_price_30,availability_30_rate
0,27886,None,0.000000
1,28871,None,0.533333
2,29051,None,0.533333
3,44391,None,0.000000
4,48373,None,0.000000
5,49552,None,1.000000
6,50263,None,1.000000
7,50515,None,1.000000
8,50523,None,1.000000
9,53921,None,0.000000


In [77]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = """
SELECT
    r.listing_id,
    COUNT(*) AS total_reviews
FROM core.review r
GROUP BY r.listing_id
"""
print(review_counts_sql)


SELECT
    r.listing_id,
    COUNT(*) AS total_reviews
FROM core.review r
GROUP BY r.listing_id



In [78]:
pd.read_sql(f"""
SELECT *
FROM ({review_counts_sql}) t
LIMIT 10
""", engine)

,listing_id,total_reviews
0,27886,311
1,28871,732
2,29051,849
3,44391,42
4,48373,5
5,49552,609
6,50263,177
7,50515,20
8,50523,563
9,53921,12


In [79]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = f"""
WITH calendar_30 AS (
    {calendar_30_sql}
),
review_counts AS (
    {review_counts_sql}
)
SELECT
    l.neighbourhood_id AS neighbourhood,
    COUNT(*) AS num_listings,
    AVG(l.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    AVG(l.minimum_nights) AS avg_minimum_nights,
    SUM(COALESCE(rc.total_reviews, 0)) AS total_reviews,
    AVG(COALESCE(rc.total_reviews, 0)) AS reviews_per_listing,
    AVG(c30.availability_30_rate) AS availability_30_rate
FROM core.listing l
LEFT JOIN calendar_30 c30
    ON l.listing_id = c30.listing_id
LEFT JOIN review_counts rc
    ON l.listing_id = rc.listing_id
GROUP BY l.neighbourhood_id
ORDER BY num_listings DESC
"""
Path("sql/01_baseline_neighbourhood_summary.sql").write_text(baseline_sql, encoding="utf-8")
print("Saved sql/01_baseline_neighbourhood_summary.sql")

Saved sql/01_baseline_neighbourhood_summary.sql


In [80]:
def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        last_df = pd.read_sql(sql, engine)
        times.append(time.perf_counter() - start)
    return last_df, times

baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(   neighbourhood  num_listings   avg_price  median_price  avg_minimum_nights  \
 0             22          1808  271.282258         240.0            3.949668   
 1              2          1207  315.880519         245.5            4.009114   
 2             20          1199  280.465331         250.0            5.464554   
 3              1           923  307.724138         240.0            4.888407   
 4              4           736  255.040816         214.5            4.240489   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        62753.0            34.708518              0.201641  
 1       106496.0            88.231980              0.290389  
 2        45931.0            38.307756              0.244287  
 3        76899.0            83.314193              0.344890  
 4        26668.0            36.233696              0.208650  ,
 [0.6365788999974029, 0.6505422000045655, 0.7694288999919081])

In [81]:
baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
display(baseline_df.head())
print("Baseline times:", baseline_times)
print("Best:", min(baseline_times), "Avg:", sum(baseline_times)/len(baseline_times))

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate
0,22,1808,271.282258,240.0,3.949668,62753.0,34.708518,0.201641
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.290389
2,20,1199,280.465331,250.0,5.464554,45931.0,38.307756,0.244287
3,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.344890
4,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.208650


Baseline times: [0.5886451000114903, 1.0838275999994949, 0.73216090000642]
Best: 0.5886451000114903 Avg: 0.801544533339135


## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [82]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

explain_baseline_sql = "EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + baseline_sql

with engine.connect() as conn:
    plan_rows = conn.execute(text(explain_baseline_sql)).fetchall()

baseline_plan_text = "\n".join(row[0] for row in plan_rows)

Path("reports/baseline_explain_analyze.txt").write_text(baseline_plan_text, encoding="utf-8")

print(baseline_plan_text[:2000])
print("\nSaved reports/baseline_explain_analyze.txt")

Sort  (cost=50871.15..50871.21 rows=22 width=180) (actual time=262.483..263.227 rows=22 loops=1)
  Sort Key: (count(*)) DESC
  Sort Method: quicksort  Memory: 27kB
  Buffers: shared hit=15712 read=5378
  ->  GroupAggregate  (cost=50633.73..50870.66 rows=22 width=180) (actual time=256.818..263.199 rows=22 loops=1)
        Group Key: l.neighbourhood_id
        Buffers: shared hit=15712 read=5378
        ->  Sort  (cost=50633.73..50659.99 rows=10506 width=53) (actual time=256.188..257.784 rows=10480 loops=1)
              Sort Key: l.neighbourhood_id
              Sort Method: quicksort  Memory: 916kB
              Buffers: shared hit=15712 read=5378
              ->  Hash Left Join  (cost=49640.42..49931.98 rows=10506 width=53) (actual time=235.882..253.249 rows=10480 loops=1)
                    Hash Cond: (l.listing_id = rc.listing_id)
                    Buffers: shared hit=15712 read=5378
                    ->  Hash Right Join  (cost=35913.49..36177.46 rows=10506 width=53) (actual t

In [83]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

explain_notes = f"""
# EXPLAIN Notes

## Observation 1
The baseline query performs aggregation over `core.calendar_day` for the next 30 days grouped by `listing_id`.
This step is expensive because it scans calendar records and computes both average price and availability rate before joining back to listings.

## Observation 2
The query separately aggregates `core.review` by `listing_id` to compute total review counts.
This introduces another full aggregation step over a large source table before the final neighbourhood-level aggregation.

## Observation 3
The final result is produced only after joining listing data with both aggregated CTEs and then grouping again by `neighbourhood_id`.
This means the same heavy work is repeated every time the dashboard query runs, which is inefficient for repeated BI reads.

## Baseline Runtime Summary
- Best runtime: {min(baseline_times):.4f} seconds
- Average runtime: {sum(baseline_times)/len(baseline_times):.4f} seconds
"""
Path("reports/explain_notes.md").write_text(explain_notes, encoding="utf-8")
print("Saved reports/explain_notes.md")

Saved reports/explain_notes.md


## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [84]:
calendar_365_sql = """
SELECT
    c.listing_id,
    AVG(CASE WHEN c.available THEN 1.0 ELSE 0.0 END) AS availability_365_rate
FROM core.calendar_day c
WHERE c.date >= CURRENT_DATE
  AND c.date < CURRENT_DATE + INTERVAL '365 days'
GROUP BY c.listing_id
"""
print(calendar_365_sql)


SELECT
    c.listing_id,
    AVG(CASE WHEN c.available THEN 1.0 ELSE 0.0 END) AS availability_365_rate
FROM core.calendar_day c
WHERE c.date >= CURRENT_DATE
  AND c.date < CURRENT_DATE + INTERVAL '365 days'
GROUP BY c.listing_id



In [85]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f"""
DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
WITH calendar_30 AS (
    {calendar_30_sql}
),
calendar_365 AS (
    {calendar_365_sql}
),
review_counts AS (
    {review_counts_sql}
)
SELECT
    l.neighbourhood_id AS neighbourhood,
    COUNT(*) AS num_listings,
    AVG(l.listing_price) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    AVG(l.minimum_nights) AS avg_minimum_nights,
    SUM(COALESCE(rc.total_reviews, 0)) AS total_reviews,
    AVG(COALESCE(rc.total_reviews, 0)) AS reviews_per_listing,
    AVG(c30.availability_30_rate) AS availability_30_rate,
    AVG(c365.availability_365_rate) AS availability_365_rate
FROM core.listing l
LEFT JOIN calendar_30 c30
    ON l.listing_id = c30.listing_id
LEFT JOIN calendar_365 c365
    ON l.listing_id = c365.listing_id
LEFT JOIN review_counts rc
    ON l.listing_id = rc.listing_id
GROUP BY l.neighbourhood_id;

CREATE INDEX IF NOT EXISTS mv_airbnb_neighbourhood_summary_neighbourhood_idx
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

CREATE INDEX IF NOT EXISTS mv_airbnb_neighbourhood_summary_num_listings_idx
ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);
"""
Path("sql/02_create_materialized_view.sql").write_text(optimized_sql, encoding="utf-8")
print("Saved sql/02_create_materialized_view.sql")


Saved sql/02_create_materialized_view.sql


In [86]:
# TODO 5.2
# Execute optimized_sql statement by statement.

optimized_statements = [
    f'DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary',
    f"""
    CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
    WITH calendar_30 AS (
        {calendar_30_sql}
    ),
    calendar_365 AS (
        {calendar_365_sql}
    ),
    review_counts AS (
        {review_counts_sql}
    )
    SELECT
        l.neighbourhood_id AS neighbourhood,
        COUNT(*) AS num_listings,
        AVG(l.listing_price) AS avg_price,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
        AVG(l.minimum_nights) AS avg_minimum_nights,
        SUM(COALESCE(rc.total_reviews, 0)) AS total_reviews,
        AVG(COALESCE(rc.total_reviews, 0)) AS reviews_per_listing,
        AVG(c30.availability_30_rate) AS availability_30_rate,
        AVG(c365.availability_365_rate) AS availability_365_rate
    FROM core.listing l
    LEFT JOIN calendar_30 c30
        ON l.listing_id = c30.listing_id
    LEFT JOIN calendar_365 c365
        ON l.listing_id = c365.listing_id
    LEFT JOIN review_counts rc
        ON l.listing_id = rc.listing_id
    GROUP BY l.neighbourhood_id
    """,
    f'''
    CREATE INDEX IF NOT EXISTS mv_airbnb_neighbourhood_summary_neighbourhood_idx
    ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood)
    ''',
    f'''
    CREATE INDEX IF NOT EXISTS mv_airbnb_neighbourhood_summary_num_listings_idx
    ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC)
    '''
]

with engine.begin() as conn:
    for stmt in optimized_statements:
        conn.execute(text(stmt))

print("Materialized view and indexes created.")

Materialized view and indexes created.


In [87]:
check_sql = f'''
SELECT *
FROM "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
ORDER BY num_listings DESC
LIMIT 10;
'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,22,1808,271.282258,240.0,3.949668,62753.0,34.708518,0.201641,0.196913
1,2,1207,315.880519,245.5,4.009114,106496.0,88.231980,0.290389,0.280129
2,20,1199,280.465331,250.0,5.464554,45931.0,38.307756,0.244287,0.232726
3,1,923,307.724138,240.0,4.888407,76899.0,83.314193,0.344890,0.318620
4,4,736,255.040816,214.5,4.240489,26668.0,36.233696,0.208650,0.204719
5,21,735,309.583908,238.0,3.993197,29444.0,40.059864,0.241179,0.236761
6,13,654,251.659509,222.0,4.061162,20448.0,31.266055,0.203823,0.194557
7,14,547,204.674330,184.0,6.076782,16461.0,30.093236,0.173796,0.159067
8,18,485,370.496032,196.5,3.336082,22623.0,46.645361,0.189966,0.186538
9,3,436,216.597403,195.0,4.399083,13988.0,32.082569,0.203976,0.192065


## 6. Compare latency

Numbers or it did not happen.

In [88]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.588645,0.801545,1.000000
1,materialized_view_read,0.361284,0.482817,1.629316


In [89]:
explain_mv_sql = "EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + dashboard_sql

with engine.connect() as conn:
    mv_plan_rows = conn.execute(text(explain_mv_sql)).fetchall()

mv_plan_text = "\n".join(row[0] for row in mv_plan_rows)
Path("reports/mv_explain_analyze.txt").write_text(mv_plan_text, encoding="utf-8")

print(mv_plan_text[:2000])
print("\nSaved reports/mv_explain_analyze.txt")

Sort  (cost=1.71..1.77 rows=22 width=180) (actual time=0.096..0.100 rows=22 loops=1)
  Sort Key: num_listings DESC
  Sort Method: quicksort  Memory: 27kB
  Buffers: shared hit=1
  ->  Seq Scan on mv_airbnb_neighbourhood_summary  (cost=0.00..1.22 rows=22 width=180) (actual time=0.019..0.063 rows=22 loops=1)
        Buffers: shared hit=1
Planning Time: 0.104 ms
Execution Time: 0.135 ms

Saved reports/mv_explain_analyze.txt


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [90]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

baseline_best = min(baseline_times)
baseline_avg = sum(baseline_times) / len(baseline_times)
dashboard_best = min(dashboard_times)
dashboard_avg = sum(dashboard_times) / len(dashboard_times)
speedup = baseline_best / dashboard_best if dashboard_best else None

report_md = f"""
# HW01_B - SQL Performance and Metabase

## Student Information
- Schema: `{STUDENT_SCHEMA}`
- Database: `qbc12_airbnb`

## Objective
The objective of this assignment was to build a performant neighbourhood-level summary for Airbnb data, measure baseline latency, optimize the workload using a materialized view, and prepare the result for dashboarding in Metabase.

## Source Tables
The source data was read from the `core` schema:
- `core.listing`
- `core.calendar_day`
- `core.review`

## Baseline Query
The baseline query joined listing data with:
- 30-day calendar aggregates
- review count aggregates

It produced the following fields:
- `neighbourhood`
- `num_listings`
- `avg_price`
- `median_price`
- `avg_minimum_nights`
- `total_reviews`
- `reviews_per_listing`
- `availability_30_rate`

Baseline SQL was saved to:
- `sql/01_baseline_neighbourhood_summary.sql`

## Baseline Runtime
- Best runtime: `{baseline_best:.4f}` seconds
- Average runtime: `{baseline_avg:.4f}` seconds

The baseline execution plan was saved to:
- `reports/baseline_explain_analyze.txt`

## Optimization
A materialized view was created in my personal schema:

- `"{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary`

This object precomputes neighbourhood-level metrics and avoids repeated heavy joins and aggregations over raw source tables.

Additional indexes were created on:
- `neighbourhood`
- `num_listings`

Optimized SQL was saved to:
- `sql/02_create_materialized_view.sql`

## Materialized View Runtime
The dashboard query read directly from the materialized view.

- Best runtime: `{dashboard_best:.4f}` seconds
- Average runtime: `{dashboard_avg:.4f}` seconds

The materialized-view execution plan was saved to:
- `reports/mv_explain_analyze.txt`

## Speedup
- Speedup vs baseline (best-to-best): `{speedup:.2f}x`

## What Changed
Compared to the baseline query, the optimized version:
1. moved repeated aggregations into a materialized view,
2. reduced dashboard-time computation,
3. made Metabase read from a prepared analytics object instead of re-running a large multi-step query.

## Metabase Dashboard
Dashboard name:
- `QBC12 HW01 - <your-github-username> - Airbnb Ops`

Required cards:
- listings by neighbourhood
- average price by neighbourhood
- review activity by neighbourhood
- availability rate by neighbourhood
- top neighbourhoods table

Screenshot path:
- `screenshots/metabase_dashboard.png`

If a shared dashboard link is required, add it here manually after creating the dashboard in Metabase.

## Deliverables
Generated files:
- `sql/01_baseline_neighbourhood_summary.sql`
- `sql/02_create_materialized_view.sql`
- `reports/baseline_explain_analyze.txt`
- `reports/explain_notes.md`
- `reports/hw01_b_sql_performance.md`
"""
Path("reports/hw01_b_sql_performance.md").write_text(report_md, encoding="utf-8")
print("Saved reports/hw01_b_sql_performance.md")

Saved reports/hw01_b_sql_performance.md


In [91]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.588645,0.801545,1.000000
1,materialized_view_read,0.361284,0.482817,1.629316


## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```